# JSRT CI / WMCI Detector for Google Colab

This notebook is prepared to run the **lung nodule candidate detector** in **Google Colab**.

It will:
- install required packages
- save the detector code into a Python file
- let you upload a JSRT `.IMG` image and optional lung mask
- run the detector
- show the overlay result
- save output files


In [ ]:
!pip -q install numpy scipy scikit-image matplotlib imageio

In [ ]:
# Save detector code to a Python file
code = r'''
"""
JSRT lung nodule candidate detector based on Convergence Index (CI) filters.

This script implements the candidate-detection part of the Hardie et al. CAD
pipeline for chest radiographs, adapted to Python and made usable on JSRT
images. The paper describes:
- resampling JRST/JSRT radiographs from 2048x2048 at 0.175 mm to 512x512 at
  0.7 mm,
- local contrast enhancement (LCE),
- a weighted / multiscale convergence-index detector,
- thresholding local maxima and thinning nearby detections.

Important note:
The paper gives the CI/WMCI equations and the window sizes, but the exact
numeric weight values of Fig. 4 are shown graphically rather than tabulated.
Accordingly, this implementation provides:
1) a faithful standard multiscale CI detector, and
2) an optional weighted mode with concentric-ring weights that you can tune.

If you want the *exact* WMCI weighting surfaces from the figure, those values
would need to be digitized from Fig. 4 or re-estimated from training data.

Reference:
R. C. Hardie et al., "Performance analysis of a new computer aided detection
system for identifying lung nodules on chest radiographs," Medical Image
Analysis, 12(3):240-258, 2008.
"""

from __future__ import annotations

import argparse
import json
import math
import os
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage as ndi
from skimage import exposure, transform
from skimage.feature import peak_local_max


@dataclass
class Detection:
    row: int
    col: int
    score: float
    radius_px: int
    scale_index: int


def read_jsrt_img(
    path: str | os.PathLike,
    shape: Tuple[int, int] = (2048, 2048),
    dtype: str = ">u2",
) -> np.ndarray:
    """Read a JSRT raw image (.IMG) as a 2D NumPy array.

    JSRT images are commonly distributed as 2048x2048 raw 12-bit grayscale
    stored in 16-bit words. Big-endian unsigned 16-bit is a common format.
    If your copy uses little-endian, change dtype to "<u2".
    """
    arr = np.fromfile(path, dtype=np.dtype(dtype))
    expected = shape[0] * shape[1]
    if arr.size != expected:
        raise ValueError(
            f"Unexpected number of pixels in {path}. Got {arr.size}, expected {expected}."
        )
    return arr.reshape(shape).astype(np.float32)


def resample_to_spacing(
    image: np.ndarray,
    original_spacing_mm: float = 0.175,
    target_spacing_mm: float = 0.7,
    order: int = 1,
) -> np.ndarray:
    """Resample image to target pixel spacing.

    In the paper, JRST images are resampled to 0.7 mm, yielding 512x512.
    """
    scale = original_spacing_mm / target_spacing_mm
    out_shape = (
        max(1, int(round(image.shape[0] * scale))),
        max(1, int(round(image.shape[1] * scale))),
    )
    return transform.resize(
        image,
        out_shape,
        order=order,
        mode="reflect",
        anti_aliasing=True,
        preserve_range=True,
    ).astype(np.float32)


def gaussian_lce(image: np.ndarray, sigma_px: float = 16.0, eps: float = 1e-6) -> np.ndarray:
    """Local contrast enhancement from the paper.

    y = (x - local_mean) / local_std
    where local mean/std are computed using a Gaussian low-pass filter.
    """
    image = image.astype(np.float32)
    local_mean = ndi.gaussian_filter(image, sigma=sigma_px, mode="reflect")
    local_sq_mean = ndi.gaussian_filter(image * image, sigma=sigma_px, mode="reflect")
    local_var = np.maximum(local_sq_mean - local_mean * local_mean, 0.0)
    local_std = np.sqrt(local_var + eps)
    return (image - local_mean) / local_std


def normalize_in_mask(image: np.ndarray, lung_mask: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Normalize image using mean/std computed only inside lung mask."""
    vals = image[lung_mask > 0]
    if vals.size == 0:
        raise ValueError("lung_mask contains no positive pixels.")
    mu = float(vals.mean())
    sigma = float(vals.std())
    return (image - mu) / (sigma + eps)


def make_disk_indices(radius_px: int) -> np.ndarray:
    """Return integer offsets (dy, dx) inside a disk excluding the center."""
    offsets = []
    for dy in range(-radius_px, radius_px + 1):
        for dx in range(-radius_px, radius_px + 1):
            if dx == 0 and dy == 0:
                continue
            if dx * dx + dy * dy <= radius_px * radius_px:
                offsets.append((dy, dx))
    return np.asarray(offsets, dtype=np.int32)


def make_ring_weight_map(radius_px: int, ring_edges: Sequence[int], ring_weights: Sequence[float]) -> np.ndarray:
    """Create a concentric-ring weighted disk for WCI experiments.

    ring_edges: ascending outer radii for each ring, e.g. [4, 7, 10]
    ring_weights: same length, e.g. [1.0, 0.8, 0.5]
    """
    if len(ring_edges) != len(ring_weights):
        raise ValueError("ring_edges and ring_weights must have the same length.")
    if ring_edges[-1] != radius_px:
        raise ValueError("The final ring edge must equal radius_px.")

    size = 2 * radius_px + 1
    yy, xx = np.mgrid[-radius_px: radius_px + 1, -radius_px: radius_px + 1]
    rr = np.sqrt(xx * xx + yy * yy)
    w = np.zeros((size, size), dtype=np.float32)

    prev = 0.0
    for edge, weight in zip(ring_edges, ring_weights):
        band = (rr > prev) & (rr <= edge)
        w[band] = float(weight)
        prev = float(edge)
    w[radius_px, radius_px] = 0.0
    return w


def ci_filter(
    lce_image: np.ndarray,
    radius_px: int,
    weight_map: Optional[np.ndarray] = None,
) -> np.ndarray:
    """Compute a convergence-index map for one scale.

    CI at each pixel is the average (or weighted sum) of the cosine between
    local gradient directions and radial directions pointing to the center.
    """
    gy, gx = np.gradient(lce_image.astype(np.float32))
    gmag = np.sqrt(gx * gx + gy * gy) + 1e-6
    ugx = gx / gmag
    ugy = gy / gmag

    offsets = make_disk_indices(radius_px)

    if weight_map is not None:
        center = radius_px
        weights = np.array([weight_map[center + dy, center + dx] for dy, dx in offsets], dtype=np.float32)
    else:
        weights = np.ones((len(offsets),), dtype=np.float32)

    if np.allclose(weights.sum(), 0.0):
        raise ValueError("weight_map produced zero total weight.")

    out = np.zeros_like(lce_image, dtype=np.float32)
    weight_sum = float(weights.sum())

    for (dy, dx), w in zip(offsets, weights):
        radial_norm = math.sqrt(dx * dx + dy * dy)
        rux = -dx / radial_norm
        ruy = -dy / radial_norm

        shifted_ugx = np.roll(ugx, shift=(dy, dx), axis=(0, 1))
        shifted_ugy = np.roll(ugy, shift=(dy, dx), axis=(0, 1))

        cos_theta = shifted_ugx * rux + shifted_ugy * ruy
        out += w * cos_theta

    out /= weight_sum

    # Invalidate borders affected by rolling.
    out[:radius_px, :] = 0.0
    out[-radius_px:, :] = 0.0
    out[:, :radius_px] = 0.0
    out[:, -radius_px:] = 0.0
    return out


def default_weight_maps(radii: Sequence[int]) -> List[Optional[np.ndarray]]:
    """Approximate WMCI-like concentric weights.

    The paper's exact Fig. 4 weight surfaces are not tabulated. These are
    reasonable concentric approximations you can tune later.
    """
    maps: List[Optional[np.ndarray]] = []
    for r in radii:
        if r <= 7:
            maps.append(None)  # standard CI disk for the smallest scale
        elif r <= 10:
            maps.append(make_ring_weight_map(r, [4, r], [1.0, 0.55]))
        else:
            maps.append(make_ring_weight_map(r, [4, 8, r], [1.0, 0.65, 0.35]))
    return maps


def wmci_detector(
    lce_image: np.ndarray,
    radii_px: Sequence[int] = (7, 10, 13),
    use_weighted: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    """Compute multiscale CI / WMCI response.

    Returns:
        wmci_max: maximum response over scales
        wmci_scale: index of the winning scale at each pixel
    """
    weight_maps = default_weight_maps(radii_px) if use_weighted else [None] * len(radii_px)
    responses = []
    for r, wm in zip(radii_px, weight_maps):
        responses.append(ci_filter(lce_image, radius_px=int(r), weight_map=wm))
    stack = np.stack(responses, axis=0)
    scale_idx = np.argmax(stack, axis=0).astype(np.int32)
    wmci_max = np.max(stack, axis=0)
    return wmci_max, scale_idx


def non_maximum_candidates(
    score_map: np.ndarray,
    scale_map: np.ndarray,
    radii_px: Sequence[int],
    threshold: float = 0.5,
    min_distance_px: int = 7,
    mask: Optional[np.ndarray] = None,
) -> List[Detection]:
    """Find local maxima above threshold and thin nearby detections.

    The paper uses a WMCI threshold of 0.5 and removes very close adjacent
    detections using a distance threshold of 5 mm. At 0.7 mm/pixel this is
    about 7 pixels.
    """
    working = score_map.copy()
    if mask is not None:
        working = np.where(mask > 0, working, -np.inf)

    peaks = peak_local_max(
        working,
        min_distance=min_distance_px,
        threshold_abs=threshold,
        exclude_border=True,
    )

    detections: List[Detection] = []
    for r, c in peaks:
        sidx = int(scale_map[r, c])
        detections.append(
            Detection(
                row=int(r),
                col=int(c),
                score=float(score_map[r, c]),
                radius_px=int(radii_px[sidx]),
                scale_index=sidx,
            )
        )

    detections.sort(key=lambda d: d.score, reverse=True)
    kept: List[Detection] = []
    for det in detections:
        keep = True
        for prev in kept:
            if math.hypot(det.row - prev.row, det.col - prev.col) < min_distance_px:
                keep = False
                break
        if keep:
            kept.append(det)
    return kept


def resize_mask(mask: np.ndarray, shape: Tuple[int, int]) -> np.ndarray:
    """Resize a binary lung mask to the detector image size."""
    out = transform.resize(
        mask.astype(np.float32),
        shape,
        order=0,
        mode="edge",
        anti_aliasing=False,
        preserve_range=True,
    )
    return out > 0.5


def load_mask(mask_path: str | os.PathLike, target_shape: Tuple[int, int]) -> np.ndarray:
    """Load a mask from .npy, .png, or image-like file and resize if needed."""
    p = Path(mask_path)
    if p.suffix.lower() == ".npy":
        mask = np.load(p)
    else:
        import imageio.v3 as iio

        mask = iio.imread(p)
        if mask.ndim == 3:
            mask = mask[..., 0]
    mask = mask > 0
    if mask.shape != target_shape:
        mask = resize_mask(mask, target_shape)
    return mask.astype(bool)


def overlay_detections(
    image: np.ndarray,
    detections: Sequence[Detection],
    out_path: str | os.PathLike,
    title: str = "CI detections",
    truth_points: Optional[Sequence[Tuple[float, float]]] = None,
) -> None:
    """Save an overlay figure showing detector outputs."""
    img = exposure.rescale_intensity(image, out_range=(0, 1))
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img, cmap="gray")

    for det in detections:
        circ = plt.Circle((det.col, det.row), det.radius_px, fill=False, linewidth=1.5)
        ax.add_patch(circ)
        ax.plot(det.col, det.row, marker="+", markersize=8)
        ax.text(det.col + 3, det.row + 3, f"{det.score:.2f}", fontsize=8)

    if truth_points is not None:
        for x, y in truth_points:
            ax.plot(x, y, marker="x", markersize=9, markeredgewidth=2)

    ax.set_title(title)
    ax.set_axis_off()
    plt.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def run_detector_on_image(
    image: np.ndarray,
    lung_mask: Optional[np.ndarray] = None,
    original_spacing_mm: float = 0.175,
    target_spacing_mm: float = 0.7,
    lce_sigma_px: float = 16.0,
    radii_px: Sequence[int] = (7, 10, 13),
    threshold: float = 0.5,
    min_distance_mm: float = 5.0,
    use_weighted: bool = True,
) -> Tuple[np.ndarray, np.ndarray, List[Detection], np.ndarray]:
    """Run JSRT preprocessing + multiscale CI candidate detection."""
    resampled = resample_to_spacing(image, original_spacing_mm, target_spacing_mm, order=1)
    lce = gaussian_lce(resampled, sigma_px=lce_sigma_px)

    resized_mask = None
    if lung_mask is not None:
        resized_mask = resize_mask(lung_mask, resampled.shape)

    wmci, scale_map = wmci_detector(lce, radii_px=radii_px, use_weighted=use_weighted)
    min_distance_px = max(1, int(round(min_distance_mm / target_spacing_mm)))
    detections = non_maximum_candidates(
        wmci,
        scale_map,
        radii_px=radii_px,
        threshold=threshold,
        min_distance_px=min_distance_px,
        mask=resized_mask,
    )
    return resampled, lce, detections, wmci


def save_detections_json(detections: Sequence[Detection], out_path: str | os.PathLike) -> None:
    """Save detections to a JSON file."""
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump([asdict(d) for d in detections], f, indent=2)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="JSRT CI/WMCI lung nodule candidate detector")
    parser.add_argument("--image", required=True, help="Path to one JSRT .IMG file")
    parser.add_argument("--mask", default=None, help="Optional lung mask (.npy/.png/...) for the image")
    parser.add_argument("--out_dir", default="ci_output", help="Output directory")
    parser.add_argument("--dtype", default=">u2", help="Raw JSRT image dtype, default='>u2'")
    parser.add_argument("--shape", default="2048,2048", help="Raw JSRT image shape, default='2048,2048'")
    parser.add_argument("--threshold", type=float, default=0.5, help="WMCI detection threshold")
    parser.add_argument("--weighted", action="store_true", help="Use weighted multiscale CI")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    h, w = [int(v) for v in args.shape.split(",")]
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    image = read_jsrt_img(args.image, shape=(h, w), dtype=args.dtype)
    mask = load_mask(args.mask, target_shape=(512, 512)) if args.mask else None

    resampled, lce, detections, wmci = run_detector_on_image(
        image=image,
        lung_mask=mask,
        original_spacing_mm=0.175,
        target_spacing_mm=0.7,
        lce_sigma_px=16.0,
        radii_px=(7, 10, 13),
        threshold=args.threshold,
        min_distance_mm=5.0,
        use_weighted=args.weighted,
    )

    stem = Path(args.image).stem
    np.save(out_dir / f"{stem}_resampled.npy", resampled)
    np.save(out_dir / f"{stem}_lce.npy", lce)
    np.save(out_dir / f"{stem}_wmci.npy", wmci)
    save_detections_json(detections, out_dir / f"{stem}_detections.json")
    overlay_detections(lce, detections, out_dir / f"{stem}_overlay.png", title=f"{stem}: CI detections")

    print(f"Saved outputs to: {out_dir}")
    print(f"Number of detections: {len(detections)}")
    for det in detections[:20]:
        print(det)


if __name__ == "__main__":
    main()

'''

with open('jsrt_ci_detector.py', 'w', encoding='utf-8') as f:
    f.write(code)

print('Saved jsrt_ci_detector.py')

In [ ]:
# Optional: mount Google Drive if your files are there
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:

# Upload your JSRT image and optional mask
from google.colab import files
uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))

# After upload, set these names
IMAGE_PATH = ''   # example: 'JPCLN001.IMG'
MASK_PATH = None  # example: 'lung_mask.npy' or 'lung_mask.png'

# If only one file was uploaded, fill IMAGE_PATH automatically
if IMAGE_PATH == '' and len(uploaded) >= 1:
    IMAGE_PATH = list(uploaded.keys())[0]

print('IMAGE_PATH =', IMAGE_PATH)
print('MASK_PATH =', MASK_PATH)


In [ ]:

# Detector parameters
DTYPE = '>u2'          # change to '<u2' if your JSRT copy is little-endian
SHAPE = (2048, 2048)
THRESHOLD = 0.5
USE_WEIGHTED = True
OUT_DIR = 'ci_output'


In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio

from jsrt_ci_detector import (
    read_jsrt_img,
    load_mask,
    run_detector_on_image,
    overlay_detections,
    save_detections_json,
)

os.makedirs(OUT_DIR, exist_ok=True)

image = read_jsrt_img(IMAGE_PATH, shape=SHAPE, dtype=DTYPE)
mask = None
if MASK_PATH is not None:
    mask = load_mask(MASK_PATH, target_shape=(512, 512))

resampled, lce, detections, wmci = run_detector_on_image(
    image=image,
    lung_mask=mask,
    original_spacing_mm=0.175,
    target_spacing_mm=0.7,
    lce_sigma_px=16.0,
    radii_px=(7, 10, 13),
    threshold=THRESHOLD,
    min_distance_mm=5.0,
    use_weighted=USE_WEIGHTED,
)

stem = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
np.save(f'{OUT_DIR}/{stem}_resampled.npy', resampled)
np.save(f'{OUT_DIR}/{stem}_lce.npy', lce)
np.save(f'{OUT_DIR}/{stem}_wmci.npy', wmci)
save_detections_json(detections, f'{OUT_DIR}/{stem}_detections.json')
overlay_detections(lce, detections, f'{OUT_DIR}/{stem}_overlay.png', title=f'{stem}: CI detections')

print('Number of detections:', len(detections))
for det in detections[:20]:
    print(det)


In [ ]:

# Show overlay result
overlay_img = iio.imread(f'{OUT_DIR}/{stem}_overlay.png')
plt.figure(figsize=(8, 8))
plt.imshow(overlay_img)
plt.axis('off')
plt.title('Detector output overlay')
plt.show()


In [ ]:

# Download outputs
from google.colab import files

for name in [
    f'{OUT_DIR}/{stem}_overlay.png',
    f'{OUT_DIR}/{stem}_detections.json',
    f'{OUT_DIR}/{stem}_resampled.npy',
    f'{OUT_DIR}/{stem}_lce.npy',
    f'{OUT_DIR}/{stem}_wmci.npy',
]:
    if os.path.exists(name):
        print('Ready:', name)


## Notes

- Use `DTYPE = '>u2'` first. If the image looks wrong, try `DTYPE = '<u2'`.
- The detector finds **candidate nodules**, not the final classifier output.
- If you also want, I can convert the **full notebook with ADT segmentation + detector** into a Colab-ready notebook too.
